In [ ]:
import os
import time
from datetime import datetime
import pandas as pd
import requests

SCOREBOARD_URL = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"
PBP_URL = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/summary"

OUT_CSV = "cbb_pbp.csv"
date = "20260228" # Use YYYYMMDD format. Change to desired date

def get_scoreboard_game_ids(date_yyyymmdd: str, group: str = "50", timeout=15) -> list[str]:
    r = requests.get(
        SCOREBOARD_URL,
        params={"dates": date_yyyymmdd, "groups": group, "limit": 500, "offset": 0},
        timeout=timeout,
    )
    r.raise_for_status()
    data = r.json()
    events = data.get("events", []) or []
    return [str(e.get("id")) for e in events if e.get("id")]

def fetch_pbp(game_id: str, timeout=20) -> dict:
    r = requests.get(PBP_URL, params={"event": game_id}, timeout=timeout)
    r.raise_for_status()
    return r.json()

def extract_player_from_text(text: str, action: str) -> str:
    if not text:
        return ""
    
    import re
    
    # Score: Everything before "makes" or "made"
    if action == "score":
        match = re.match(r"^(.+?)\s+makes?\b", text, re.IGNORECASE)
        if match:
            return match.group(1).strip()
    
    # Miss: Everything before "misses"
    elif action == "miss":
        match = re.match(r"^(.+?)\s+misses?\b", text, re.IGNORECASE)
        if match:
            return match.group(1).strip()
    
    # Rebound: "Player Name Rebound"
    elif action == "rebound":
        if "rebound" in text.lower():
            # Extract name before "Rebound", removing offensive/defensive 
            match = re.match(r"([^(]+?)\s+(?:Offensive|Defensive)?\s*Rebound", text, re.IGNORECASE)
            if match:
                name = match.group(1).strip()
                # Remove offensive/defensive from the name if it got included
                name = re.sub(r"\s+(?:Offensive|Defensive)$", "", name, flags=re.IGNORECASE)
                return name
    
    # Assist: Everything in parenthesis before "assists"
    elif action == "assist":
        if "assist" in text.lower():
            # Extract what's in parenthesis before "assists"
            match = re.search(r"\((.+?)\s+assists?\)", text, re.IGNORECASE)
            if match:
                return match.group(1).strip()
    
    return ""

def extract_rebound_type(text: str) -> str:
    if not text or "rebound" not in text.lower():
        return ""
    
    import re
    
    match = re.search(r"(Offensive|Defensive)\s+Rebound", text, re.IGNORECASE)
    if match:
        return match.group(1)
    return ""

def flatten_plays(game_id: str, pbp_json: dict) -> pd.DataFrame:
    plays = pbp_json.get("plays", []) or []
    rows = []
    
    # Extract home and away team info once for the entire game
    home_team = None
    away_team = None
    team_id_to_name = {}
    
    boxscore = pbp_json.get("boxscore", {})
    teams = boxscore.get("teams", [])
    
    # Get team names and IDs from boxscore teams with homeAway designation
    for team in teams:
        team_info = team.get("team", {})
        team_name = team_info.get("location") or team_info.get("name")
        team_id = team_info.get("id")
        home_away = team.get("homeAway", "").lower()
        
        if home_away == "home":
            home_team = team_name
            if team_id:
                team_id_to_name[str(team_id)] = team_name
        elif home_away == "away":
            away_team = team_name
            if team_id:
                team_id_to_name[str(team_id)] = team_name

    for play_num, p in enumerate(plays, start=1):
        period = p.get("period", {}).get("number") if isinstance(p.get("period"), dict) else p.get("period")
        clock = p.get("clock", {}).get("displayValue") if isinstance(p.get("clock"), dict) else p.get("clock")
        text = p.get("text")
        home_score = p.get("homeScore")
        away_score = p.get("awayScore")

        # Get team_id from play
        team_id = None
        if isinstance(p.get("team"), dict):
            team_id = p["team"].get("id")
        
        # Map team_id to team name using our lookup
        team = team_id_to_name.get(str(team_id)) if team_id else None

        # Some play entries include a type
        play_type = None
        play_type_id = None
        if isinstance(p.get("type"), dict):
            play_type = p["type"].get("text")
            play_type_id = p["type"].get("id")

        # Extract player actions from text
        scorer = extract_player_from_text(text, "score") if p.get("scoreValue") else ""
        missser = extract_player_from_text(text, "miss") if "miss" in (text or "").lower() else ""
        rebounder = extract_player_from_text(text, "rebound") if "rebound" in (text or "").lower() else ""
        rebound_type = extract_rebound_type(text) if "rebound" in (text or "").lower() else ""
        assister = extract_player_from_text(text, "assist") if "assist" in (text or "").lower() else ""

        # Extract coordinates of location on court
        coord = p.get("coordinate", {})
        coord_x = coord.get("x") if isinstance(coord, dict) else None
        coord_y = coord.get("y") if isinstance(coord, dict) else None

        rows.append({
            "game_id": game_id,
            "play_number": play_num,
            "period": period,
            "clock": clock,
            "text": text,
            "home_team": home_team,
            "away_team": away_team,
            "home_score": home_score,
            "away_score": away_score,
            "team_id": team_id,
            "team": team,
            "play_type_id": play_type_id,
            "play_type": play_type,
            "score": scorer,
            "miss": missser,
            "assist": assister,
            "rebound": rebounder,
            "rebound_type": rebound_type,
            "coord_x": coord_x,
            "coord_y": coord_y,
            "wallclock": p.get("wallclock") or p.get("startTime") or None,
        })

    return pd.DataFrame(rows)

def append_df_to_csv(df: pd.DataFrame, path: str):
    write_header = not os.path.exists(path)
    df.to_csv(path, mode="a", header=write_header, index=False)

def run(date_yyyymmdd: str, group: str = "50", sleep_s: float = 0.25):
    game_ids = get_scoreboard_game_ids(date_yyyymmdd, group=group)
    print(f"Found {len(game_ids)} games for {date_yyyymmdd} (groups={group})")

    for i, game_id in enumerate(game_ids, start=1):
        try:
            pbp_json = fetch_pbp(game_id)
            df = flatten_plays(game_id, pbp_json)
            if not df.empty:
                append_df_to_csv(df, OUT_CSV)
            print(f"[{i}/{len(game_ids)}] saved {len(df)} plays for game_id={game_id}")
        except requests.RequestException as e:
            print(f"[{i}/{len(game_ids)}] ERROR game_id={game_id}: {e}")
        time.sleep(sleep_s)

if __name__ == "__main__":
    run(date, group="50")

Found 143 games for 20260228 (groups=50)
[1/143] saved 446 plays for game_id=401820771
[2/143] saved 458 plays for game_id=401827707
[3/143] saved 472 plays for game_id=401827712
[4/143] saved 442 plays for game_id=401827711
[5/143] saved 411 plays for game_id=401822963
[128/143] saved 467 plays for game_id=401829270
[129/143] saved 496 plays for game_id=401829266
[130/143] saved 420 plays for game_id=401827709
[131/143] saved 366 plays for game_id=401823833
[132/143] saved 497 plays for game_id=401823270
